In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# Unsupervised Clustering Project

This notebook provides a boilerplate structure for an unsupervised clustering project. It includes sections for data input, model training, and visualization of the results.

In [2]:
from typing import List, Optional, Sequence, Dict, Iterable
from __future__ import annotations
from dataclasses import dataclass

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans # Example clustering algorithm
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import dates as mdates
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import math

from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


## 1. Data Input

Specify the path to your parquet data file and load it into a pandas DataFrame.

In [3]:
def lazy_sample_parquet(
    path: str,
    fraction: float = 0.05,
    seed: int = 42,
    columns: list[str] | None = None,
    stratify_col: str | None = None,
) -> pl.DataFrame:
    """
    Low-memory sampling loader that works even if LazyFrame.sample() isn't available.
    - Loads a small random subset of rows directly from disk.
    - fraction = fraction of total rows to load (0–1)
    - columns = list of columns to read (for column pruning)
    - stratify_col = optional column to preserve distribution (lightweight, approximate)
    """
    # Step 1: Read just the schema to see what’s available
    schema = pl.read_parquet(path, n_rows=0).schema
    available_cols = list(schema.keys())
    if columns:
        columns = [c for c in columns if c in available_cols]
    else:
        columns = available_cols

    # Step 2: If stratification requested, read that column separately (small)
    if stratify_col and stratify_col in available_cols:
        small_col = pl.read_parquet(path, columns=[stratify_col])
        unique_vals = small_col[stratify_col].unique()
        n_groups = len(unique_vals)
        print(f"Stratifying by '{stratify_col}' across {n_groups:,} groups.")

        # Take a small fraction from each group (roughly)
        sampled_groups = []
        for val in unique_vals:
            mask = small_col[stratify_col] == val
            idx = np.where(mask)[0]
            n_take = max(1, int(len(idx) * fraction))
            chosen_idx = np.random.default_rng(seed).choice(idx, size=n_take, replace=False)
            sampled_groups.append(chosen_idx)
        take_indices = np.concatenate(sampled_groups)
        # This is only feasible if your file isn't > few hundred MBs
        return pl.read_parquet(path, columns=columns, row_indices=take_indices.tolist())

    # Step 3: No stratification → use chunked row-group sampling
    # Count total row groups without loading all data
    meta = pl.read_parquet(path, n_rows=0)
    # Determine rough number of rows per group (approx)
    total_rows = None
    try:
        # Some Polars versions expose this
        total_rows = meta.shape[0]
    except Exception:
        pass

    # Simple fallback: random row group fraction
    print(f"Sampling ≈{fraction*100:.1f}% of rows (columns={len(columns)})")
    df = pl.read_parquet(path, columns=columns)
    n = df.height
    take_n = int(max(1, n * fraction))
    idx = np.random.default_rng(seed).choice(n, take_n, replace=False)
    sample_df = df[idx]
    print(f"Loaded {sample_df.height:,} rows out of {n:,}")
    return sample_df


## 2. Data Preprocessing and Training

Perform necessary data preprocessing steps (e.g., scaling, feature selection) and then train your chosen unsupervised clustering model.

## Split the data for supervised / unsupervised training

In [4]:
def split_for_supervised_unsupervised(
    df: pl.DataFrame,
    test_size: float = 0.5,
    random_state: int = 42
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Randomly split a Polars DataFrame into two disjoint parts:
    one for supervised and one for unsupervised learning.
    """
    # Convert just indices for speed & reproducibility
    indices = np.arange(df.height)
    idx_train, idx_unsup = train_test_split(
        indices, test_size=test_size, random_state=random_state, shuffle=True
    )

    df_supervised = df[idx_train]
    df_unsupervised = df[idx_unsup]

    print(f"Split complete:")
    print(f"  Supervised subset:   {df_supervised.height:,} rows")
    print(f"  Unsupervised subset: {df_unsupervised.height:,} rows")

    return df_supervised, df_unsupervised


### PCA

In [5]:
def _prep_features(
    df: pl.DataFrame,
    feature_cols: List[str],
    standardize: bool = True,
    dtype: np.dtype = np.float32,
) -> np.ndarray:
    """
    Extract features from a Polars DF, fill nulls, cast to dtype, and (optionally) standardize.
    Returns a NumPy array without round-tripping through pandas.
    """
    X = (
        df.select([pl.col(c).cast(pl.Float32).fill_null(0.0) for c in feature_cols])
          .to_numpy()  # zero-copy from Polars to NumPy when possible
          .astype(dtype, copy=False)
    )
    if standardize:
        X = StandardScaler(copy=False).fit_transform(X)  # in-place when possible
    return X

def apply_pca_preprocessing(
    df: pl.DataFrame,
    numeric_features: list[str],
    n_components: float = 0.95,
    standardize: bool = True,
) -> tuple[pl.DataFrame, PCA, list[str]]:
    """
    Apply PCA in NumPy space and append components back to the Polars DF.
    Avoid pandas, use float32.
    """
    X = _prep_features(df, numeric_features, standardize=standardize, dtype=np.float32)
    pca = PCA(n_components=n_components, svd_solver="full", random_state=42)
    Z = pca.fit_transform(X).astype(np.float32, copy=False)

    pca_cols = [f"pca_component_{i+1}" for i in range(Z.shape[1])]
    pca_df = pl.DataFrame(Z, schema=pca_cols)          # zero-copy into Polars
    result_df = pl.concat([df, pca_df], how="horizontal")

    print(f"PCA reduced {len(numeric_features)} → {len(pca_cols)} comps "
          f"(explained={pca.explained_variance_ratio_.sum():.3f})")
    return result_df, pca, pca_cols

In [6]:

def perform_clustering_from_X(
    X: np.ndarray,
    df: pl.DataFrame,
    n_clusters: int,
    random_state: int = 42,
    n_init: int | str = "auto",
) -> pl.DataFrame:
    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=n_init,
        algorithm="elkan",   # faster for euclidean
    )
    labels = km.fit_predict(X)
    print(f"KMeans inertia: {km.inertia_:.2f}")
    return df.with_columns(pl.Series("cluster", labels))


def perform_clustering(df: pl.DataFrame, feature_cols: List[str], n_clusters: int = 5) -> pl.DataFrame:
    X = _prep_features(df, feature_cols, standardize=True, dtype=np.float32)
    out = perform_clustering_from_X(X, df, n_clusters=n_clusters)
    # Optional: distribution
    counts = out.group_by("cluster").len().sort("cluster")
    print("\nCluster distribution:")
    for row in counts.iter_rows():
        cid, cnt = row
        print(f"  Cluster {cid}: {cnt:,} points ({cnt / df.height * 100:.1f}%)")
    return out


#%%
def analyze_clusters(df: pl.DataFrame, feature_cols: List[str], cluster_col: str = "cluster") -> pl.DataFrame:
    """
    Analyze cluster characteristics

    Args:
        df: DataFrame with cluster assignments
        feature_cols: Features used for clustering
        cluster_col: Name of cluster column

    Returns:
        DataFrame with cluster statistics
    """

    print(f"\nAnalyzing clusters using features: {feature_cols}")

    # Calculate cluster statistics
    cluster_stats = df.group_by(cluster_col).agg([
        pl.len().alias("count"),
        *[pl.col(col).mean().alias(f"{col}_mean") for col in feature_cols],
        *[pl.col(col).std().alias(f"{col}_std") for col in feature_cols],
        *[pl.col(col).min().alias(f"{col}_min") for col in feature_cols],
        *[pl.col(col).max().alias(f"{col}_max") for col in feature_cols]
    ]).sort(cluster_col)

    print("\nCluster Statistics:")
    print(cluster_stats)

    return cluster_stats


In [7]:
def _nice_grid(n):
    if n <= 1: return (1, 1)
    cols = int(np.ceil(np.sqrt(n)))
    rows = int(np.ceil(n / cols))
    return rows, cols

def to_pandas_minimal(df: pl.DataFrame, cols: list[str], label_col: str) -> pd.DataFrame:
    """
    Polars -> pandas for a small column subset:
      • dedupe requested names (order preserved)
      • tolerate duplicate column names
      • cast label to categorical
    """
    # 1) dedupe requested names
    cols = list(dict.fromkeys(cols))

    # 2) select and convert (no 'copy=' — not supported in some pyarrow versions)
    #    Optionally disable extension arrays to keep dtypes simple.
    out = df.select(cols).to_pandas(use_pyarrow_extension_array=False)

    # 3) if upstream duplicates slipped in, drop dup columns
    if out.columns.duplicated().any():
        out = out.loc[:, ~out.columns.duplicated()].copy()

    # 4) make label categorical
    if label_col in out.columns:
        out[label_col] = pd.Categorical(out[label_col])

    return out


def build_plot_cols(feature_set: list[str], label_col: str = "cluster", max_feats: int = 8) -> list[str]:
    """label first, then up to max_feats features (excluding label if present)."""
    feats = [c for c in feature_set if c != label_col][:max_feats]
    return [label_col] + feats


def regime_dashboard_windows(
    df: pd.DataFrame,
    label_col: str,
    candidate_columns=None,
    max_charts: int = 8,
    title_prefix: str = "Regime Characteristics Over Time",
    palette=None,
    include_object_date_guess: bool = False,
):
    if label_col not in df.columns:
        raise ValueError(f"label_col '{label_col}' not found in dataframe.")

    # Candidate axes
    if candidate_columns is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]", "datetimetz"]).columns.tolist()
        obj_date_like = []
        if include_object_date_guess:
            for c in df.select_dtypes(include=["object"]).columns.tolist():
                try:
                    pd.to_datetime(df[c].dropna().astype(str).head(5), errors="raise")
                    obj_date_like.append(c)
                except Exception:
                    pass
        candidate_columns = list(dict.fromkeys(dt_cols + obj_date_like + numeric_cols))
    if not candidate_columns:
        raise ValueError("No candidate columns found. Provide candidate_columns or check your dataframe dtypes.")

    # Do not allow the label column to be used as X/Y axis
    axis_options = [c for c in candidate_columns if c != label_col]
    if not axis_options:
        raise ValueError("No numeric/datetime columns available for axes (after excluding label column).")

    # Regime colors
    regimes = pd.Categorical(df[label_col])
    cats = list(regimes.categories)
    if palette is None:
        base = plt.cm.tab10.colors if len(cats) <= 10 else plt.cm.tab20.colors
        color_map = {cat: base[i % len(base)] for i, cat in enumerate(cats)}
    else:
        base = plt.cm.tab10.colors
        color_map = {cat: palette.get(cat, base[i % len(base)]) for i, cat in enumerate(cats)}

    # --- Widgets ---
    n_charts = widgets.IntSlider(value=min(4, max(1, len(axis_options)//2)),
                                 min=1, max=max(1, max_charts), step=1,
                                 description="Windows", continuous_update=False)
    width_slider  = widgets.IntSlider(value=15, min=8, max=24, step=1, description="Fig width",  continuous_update=False)
    height_slider = widgets.IntSlider(value=10, min=6, max=20, step=1, description="Fig height", continuous_update=False)
    size_slider   = widgets.IntSlider(value=20, min=5, max=80, step=1, description="Marker size", continuous_update=False)
    alpha_slider  = widgets.FloatSlider(value=0.6, min=0.05, max=1.0, step=0.05, description="Alpha", continuous_update=False)
    title_text    = widgets.Text(value=title_prefix, description="Suptitle", continuous_update=False)

    max_xticks    = widgets.IntSlider(value=6, min=3, max=15, step=1, description="Max X ticks", continuous_update=False)
    rotate_xticks = widgets.IntSlider(value=30, min=0, max=90, step=5, description="Rotate X°", continuous_update=False)
    concise_dates = widgets.Checkbox(value=True, description="Concise datetime format")

    selectors_box = widgets.VBox()

    def make_xy_row(i):
        return widgets.HBox([
            widgets.Dropdown(options=axis_options,
                             value=axis_options[min(i, len(axis_options)-1)],
                             description=f"X{i+1}", layout=widgets.Layout(width="50%")),
            widgets.Dropdown(options=axis_options,
                             value=axis_options[(i+1) % len(axis_options)],
                             description=f"Y{i+1}", layout=widgets.Layout(width="50%")),
        ])

    def refresh_selectors(*_):
        selectors_box.children = [make_xy_row(i) for i in range(n_charts.value)]

    n_charts.observe(lambda change: refresh_selectors(), names="value")
    refresh_selectors()

    out = widgets.Output()

    def _is_datetime_series(s: pd.Series) -> bool:
        return pd.api.types.is_datetime64_any_dtype(s)

    def _format_axes(ax, x_series: pd.Series):
        if _is_datetime_series(x_series):
            locator = mdates.AutoDateLocator(minticks=3, maxticks=max_xticks.value)
            formatter = mdates.ConciseDateFormatter(locator) if concise_dates.value else mdates.AutoDateFormatter(locator)
            ax.xaxis.set_major_locator(locator)
            ax.xaxis.set_major_formatter(formatter)
        else:
            ax.xaxis.set_major_locator(MaxNLocator(nbins=max_xticks.value, prune='both'))
        for tick in ax.get_xticklabels():
            tick.set_rotation(rotate_xticks.value)
            tick.set_horizontalalignment("right" if rotate_xticks.value else "center")

    def render(*_):
        with out:
            clear_output(wait=True)

            k = n_charts.value
            rows, cols = _nice_grid(k)
            fig, axes = plt.subplots(rows, cols, figsize=(width_slider.value, height_slider.value))
            axes_list = [axes] if rows*cols == 1 else np.array(axes).reshape(-1).tolist()

            for i in range(k):
                ax = axes_list[i]
                x_col = selectors_box.children[i].children[0].value
                y_col = selectors_box.children[i].children[1].value

                # Deduplicate selected cols (prevents duplicate column name issues)
                cols_sel = [x_col, y_col, label_col]
                cols_sel = list(dict.fromkeys(cols_sel))
                valid = df[cols_sel].dropna()

                # Ensure label is a Series even if duplicate names exist upstream
                lab = valid[label_col]
                if isinstance(lab, pd.DataFrame):
                    lab = lab.iloc[:, 0]

                # Draw per-regime
                for cat in cats:
                    mask = (lab == cat)
                    if not mask.any():
                        continue
                    sub = valid.loc[mask]
                    ax.scatter(
                        sub[x_col].values, sub[y_col].values,
                        s=size_slider.value, alpha=alpha_slider.value,
                        c=[color_map[cat]], label=str(cat)
                    )

                ax.set_title(f"{x_col} vs {y_col}")
                ax.set_xlabel(x_col)
                ax.set_ylabel(y_col)
                ax.grid(True, alpha=0.3)

                # Apply smart tick formatting
                _format_axes(ax, valid[x_col])

            # Hide unused
            for j in range(k, rows*cols):
                axes_list[j].axis("off")

            # Shared legend
            fig.suptitle(f"{title_text.value} — {label_col}", y=0.995)
            handles, labels = [], []
            for ax in axes_list[:k]:
                h, l = ax.get_legend_handles_labels()
                handles += h; labels += l
            if handles:
                seen, h_u, l_u = set(), [], []
                for h, l in zip(handles, labels):
                    if l not in seen:
                        seen.add(l); h_u.append(h); l_u.append(l)
                fig.legend(
                    h_u, l_u, title=str(label_col),
                    loc="upper right", ncol=min(len(l_u), 5), bbox_to_anchor=(1, 1)
                )

            fig.tight_layout(rect=(0, 0, 1, 0.97))
            plt.subplots_adjust(bottom=0.12)
            plt.show()

    render_btn = widgets.Button(description="Render", button_style="primary", icon="refresh")
    render_btn.on_click(lambda _: render())

    controls_top = widgets.HBox([n_charts, size_slider, alpha_slider])
    controls_mid = widgets.HBox([max_xticks, rotate_xticks, concise_dates])
    controls_bottom = widgets.HBox([width_slider, height_slider, title_text])
    ui = widgets.VBox([controls_top, controls_mid, controls_bottom, selectors_box, render_btn, out])
    display(ui)
    return ui


In [8]:
@dataclass
class KScanResult:
    ks: List[int]
    inertia: List[float]
    silhouette: List[Optional[float]]
    calinski_harabasz: List[Optional[float]]
    davies_bouldin: List[Optional[float]]
    best_k: int
    best_by: Dict[str, int]   # per-metric winners
    w_distance: Optional[List[Optional[float]]] = None



## 3. Visualization

Visualize the clustering results to understand the formed clusters.

In [9]:
def plot_k_scan(res: KScanResult) -> None:
    """Render diagnostic plots for k selection."""
    ks = np.array(res.ks)

    # Inertia (Elbow)
    plt.figure(figsize=(7,5))
    plt.plot(ks, res.inertia, marker="o")
    plt.title("Elbow (Inertia) vs k")
    plt.xlabel("k")
    plt.ylabel("Inertia (lower is better)")
    plt.grid(True, alpha=0.3)
    plt.axvline(res.best_by.get("elbow_inertia", None), linestyle="--", alpha=0.6)
    plt.show()

    # Silhouette
    if any(v is not None for v in res.silhouette):
        plt.figure(figsize=(7,5))
        plt.plot(ks, [np.nan if v is None else v for v in res.silhouette], marker="o")
        plt.title("Silhouette Score vs k (higher is better)")
        plt.xlabel("k")
        plt.ylabel("Silhouette")
        plt.grid(True, alpha=0.3)
        if "silhouette" in res.best_by:
            plt.axvline(res.best_by["silhouette"], linestyle="--", alpha=0.6)
        plt.show()

    # Calinski–Harabasz
    if any(v is not None for v in res.calinski_harabasz):
        plt.figure(figsize=(7,5))
        plt.plot(ks, [np.nan if v is None else v for v in res.calinski_harabasz], marker="o")
        plt.title("Calinski–Harabasz Index vs k (higher is better)")
        plt.xlabel("k")
        plt.ylabel("Calinski–Harabasz")
        plt.grid(True, alpha=0.3)
        if "calinski_harabasz" in res.best_by:
            plt.axvline(res.best_by["calinski_harabasz"], linestyle="--", alpha=0.6)
        plt.show()

    # Davies–Bouldin
    if any(v is not None for v in res.davies_bouldin):
        plt.figure(figsize=(7,5))
        plt.plot(ks, [np.nan if v is None else v for v in res.davies_bouldin], marker="o")
        plt.title("Davies–Bouldin Index vs k (lower is better)")
        plt.xlabel("k")
        plt.ylabel("Davies–Bouldin")
        plt.grid(True, alpha=0.3)
        if "davies_bouldin" in res.best_by:
            plt.axvline(res.best_by["davies_bouldin"], linestyle="--", alpha=0.6)
        plt.show()

    if any(v is not None for v in res.silhouette) and any(v is not None for v in res.davies_bouldin):
        if res.w_distance is not None:
            wdist = np.array([np.nan if v is None else v for v in res.w_distance], dtype=float)
        else:
            # Fallback: recompute safely if older result object
            silh_vals = np.array([np.nan if v is None else float(v) for v in res.silhouette])
            dbs_vals  = np.array([np.nan if v is None else float(v) for v in res.davies_bouldin])
            valid = np.isfinite(silh_vals) & np.isfinite(dbs_vals)
            wdist = np.full_like(silh_vals, np.nan, dtype=float)
            if np.any(valid):
                s_min, s_max = np.nanmin(silh_vals[valid]), np.nanmax(silh_vals[valid])
                d_min, d_max = np.nanmin(dbs_vals[valid]),  np.nanmax(dbs_vals[valid])
                s_den = (s_max - s_min) if (s_max - s_min) > 0 else 1.0
                d_den = (d_max - d_min) if (d_max - d_min) > 0 else 1.0
                s_n = np.full_like(silh_vals, np.nan, dtype=float); s_n[valid]=(silh_vals[valid]-s_min)/s_den
                d_n = np.full_like(dbs_vals,  np.nan, dtype=float); d_n[valid]=(dbs_vals[valid]-d_min)/d_den
                wdist[valid] = np.sqrt((1 - s_n[valid])**2 + d_n[valid]**2)

        plt.figure(figsize=(7,5))
        plt.plot(np.array(res.ks), wdist, marker="o")
        plt.title("W-distance (Silhouette + Davies–Bouldin) vs k (lower is better)")
        plt.xlabel("k"); plt.ylabel("W-distance"); plt.grid(True, alpha=0.3)
        if "w_distance" in res.best_by:
            plt.axvline(res.best_by["w_distance"], linestyle="--", alpha=0.6)
        plt.show()

    # Summary
    print("Per-metric winners:", res.best_by)
    print(f"Suggested k (ensemble): {res.best_k}")

## 4. Choosing k (model selection for KMeans)
We'll scan k over a range and compute multiple unsupervised criteria:
- Inertia (Elbow)
- Silhouette Score (higher is better; requires k >= 2)
- Calinski–Harabasz (higher is better)
- Davies–Bouldin (lower is better)
We'll then plot the metrics and propose a "best k" based on an ensemble heuristic.



In [10]:

def _kneedle(x: np.ndarray, y: np.ndarray) -> int:
    """
    Very small 'knee' detector for the Elbow curve (inertia).
    Returns index of knee in y(x) by maximizing distance to line between endpoints.
    """
    # Normalize x, y to [0,1] to be scale-invariant
    x_n = (x - x.min()) / (x.max() - x.min() + 1e-12)
    y_n = (y - y.min()) / (y.max() - y.min() + 1e-12)

    p1 = np.array([x_n[0], y_n[0]])
    p2 = np.array([x_n[-1], y_n[-1]])
    v = p2 - p1
    v /= np.linalg.norm(v) + 1e-12

    # Per-point distance to line p1->p2
    dists = []
    for i in range(len(x_n)):
        p = np.array([x_n[i], y_n[i]])
        proj_len = np.dot(p - p1, v)
        proj = p1 + proj_len * v
        d = np.linalg.norm(p - proj)
        dists.append(d)
    return int(np.argmax(dists))


from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
from sklearn.cluster import MiniBatchKMeans

def scan_k(
    df: pl.DataFrame | None,
    feature_cols: List[str] | None,
    k_range: Iterable[int] = range(2, 13),
    *,
    X: np.ndarray | None = None,   # NEW: precomputed feature matrix
    standardize: bool = True,
    random_state: int = 42,
    n_init: int | str = "auto",
    w_alpha: float = 1.0,
    w_beta: float = 1.0,
    n_jobs: int = 1,
    silhouette_sample_size: Optional[int] = None,
    use_minibatch: bool = False,
    mbk_max_iter: int = 100,
) -> KScanResult:
    if X is None:
        assert df is not None and feature_cols is not None, "Provide df+feature_cols or X."
        X = _prep_features(df, feature_cols, standardize=standardize, dtype=np.float32)
    else:
        # assume X is already standardized/float32
        pass

    ks = list(k_range)
    KM = MiniBatchKMeans if use_minibatch else KMeans
    km_kwargs = dict(n_clusters=None, random_state=random_state, n_init=n_init)
    if not use_minibatch:
        km_kwargs["algorithm"] = "elkan"
    else:
        km_kwargs.update(dict(batch_size=4096, max_iter=mbk_max_iter, reassignment_ratio=0.01))

    def _one_k(k: int):
        # Avoid nested parallelism: give BLAS/OpenMP 1 thread inside each job
        with threadpool_limits(limits=1):
            km = KM(**{**km_kwargs, "n_clusters": k})
            labels = km.fit_predict(X)
            inertia = float(km.inertia_)

            # metrics (guard degenerate labelings)
            if k >= 2 and len(set(labels)) > 1:
                sil = float(
                    silhouette_score(
                        X, labels,
                        sample_size=min(silhouette_sample_size, len(X)) if silhouette_sample_size else None,
                        random_state=random_state
                    )
                )
                ch  = float(calinski_harabasz_score(X, labels))
                db  = float(davies_bouldin_score(X, labels))
            else:
                sil = ch = db = None

            return k, inertia, sil, ch, db

    if n_jobs == 1:
        results = [_one_k(k) for k in ks]
    else:
        # threads, not processes (shared memory; no X pickling)
        results = Parallel(
            n_jobs=n_jobs,
            prefer="threads",
            require="sharedmem",
            verbose=10,
        )(delayed(_one_k)(k) for k in ks)
    # unpack in k order
    results = sorted(results, key=lambda t: t[0])
    inertia = [r[1] for r in results]
    silh    = [r[2] for r in results]
    chs     = [r[3] for r in results]
    dbs     = [r[4] for r in results]

    best_by: Dict[str, int] = {}
    knee_idx = _kneedle(np.array(ks), np.array(inertia))
    best_by["elbow_inertia"] = ks[knee_idx]

    if any(v is not None for v in silh):
        vals = np.array([v if v is not None else -np.inf for v in silh])
        best_by["silhouette"] = ks[int(np.nanargmax(vals))]

    if any(v is not None for v in chs):
        vals = np.array([v if v is not None else -np.inf for v in chs])
        best_by["calinski_harabasz"] = ks[int(np.nanargmax(vals))]

    if any(v is not None for v in dbs):
        vals = np.array([v if v is not None else np.inf for v in dbs])
        best_by["davies_bouldin"] = ks[int(np.nanargmin(vals))]

    wdist_list: Optional[List[Optional[float]]] = None
    if any(v is not None for v in silh) and any(v is not None for v in dbs):
        silh_arr = np.array([np.nan if v is None else float(v) for v in silh], dtype=float)
        dbs_arr  = np.array([np.nan if v is None else float(v) for v in dbs], dtype=float)
        valid = np.isfinite(silh_arr) & np.isfinite(dbs_arr)
        if np.any(valid):
            s_min, s_max = np.nanmin(silh_arr[valid]), np.nanmax(silh_arr[valid])
            d_min, d_max = np.nanmin(dbs_arr[valid]),  np.nanmax(dbs_arr[valid])
            s_den = (s_max - s_min) if (s_max - s_min) > 0 else 1.0
            d_den = (d_max - d_min) if (d_max - d_min) > 0 else 1.0
            silh_n = np.full_like(silh_arr, np.nan); silh_n[valid] = (silh_arr[valid]-s_min)/s_den
            dbs_n  = np.full_like(dbs_arr,  np.nan); dbs_n[valid]  = (dbs_arr[valid]-d_min)/d_den
            wdist = np.full_like(silh_arr, np.nan, dtype=float)
            wdist[valid] = np.sqrt((w_alpha*(1 - silh_n[valid]))**2 + (w_beta*dbs_n[valid])**2)
            wdist_list = [None if np.isnan(x) else float(x) for x in wdist]
            best_by["w_distance"] = ks[int(np.nanargmin(wdist))]

    votes = list(best_by.values())
    counts = pd.Series(votes).value_counts()
    best_k = int(counts.index[np.argmax(counts.values)])
    tied = counts[counts == counts.max()].index.tolist()
    if len(tied) > 1:
        best_k = int(min(tied))

    return KScanResult(
        ks=ks, inertia=inertia, silhouette=silh, calinski_harabasz=chs,
        davies_bouldin=dbs, best_k=best_k, best_by=best_by, w_distance=wdist_list
    )




def choose_k_and_cluster(
    df: pl.DataFrame,
    feature_cols: List[str],
    k_range: Iterable[int] = range(2, 13),
    standardize: bool = True,
    random_state: int = 42,
    n_init: int = 10,
    verbose: bool = True,
) -> tuple[pl.DataFrame, KScanResult]:
    """
    Convenience wrapper:
      1) Scan k
      2) Plot diagnostics
      3) Cluster with chosen k and return new DF + scan result
    """
    res = scan_k(
        df=df,
        feature_cols=feature_cols,
        k_range=k_range,
        standardize=standardize,
        random_state=random_state,
        n_init=n_init,
    )
    plot_k_scan(res)
    best_k = res.best_k

    if verbose:
        print(f"\nFitting final KMeans with k = {best_k}")

    X = _prep_features(df, feature_cols, standardize=standardize)
    km = KMeans(n_clusters=best_k, random_state=random_state, n_init=n_init)
    labels = km.fit_predict(X)

    out_df = df.with_columns(pl.Series("cluster", labels))
    return out_df, res

def auto_k_range(df: pl.DataFrame, feature_cols: list[str], max_cap: int = 13) -> range:
    """
    Automatically choose a reasonable k range based on dataset size and feature count.

    Heuristic:
      - Start at k = 2
      - Upper bound grows with sqrt(n_samples) but capped by both:
          * number of features (cannot exceed informative dimensions)
          * max_cap (default = 20)

    Returns:
        range object usable as k_range
    """
    n_samples = df.height
    n_features = len(feature_cols)

    # Use square-root heuristic (typical for exploratory clustering)
    upper = int(min(max_cap, max(3, math.sqrt(n_samples / 100)), n_features * 2))
    upper = max(upper, 4)  # ensure we scan at least up to 4

    print(f"Auto-selected k_range = range(2, {upper}) "
          f"(samples={n_samples:,}, features={n_features})")
    return range(2, upper)

## Example usage
- Option A: Use your raw features (e.g., implied_volatility, delta, gamma)
- Option B: Use PCA components to choose k in the reduced space.

In [11]:
from typing import Tuple, Iterable, Optional
from sklearn.metrics import adjusted_rand_score
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits

def _normalize_metrics(silh, dbs):
    print("Normalizing silhouette and Davies–Bouldin metrics...")
    silh_arr = np.array([np.nan if v is None else float(v) for v in silh], dtype=float)
    dbs_arr  = np.array([np.nan if v is None else float(v) for v in dbs], dtype=float)
    valid = np.isfinite(silh_arr) & np.isfinite(dbs_arr)
    s_n = np.full_like(silh_arr, np.nan, dtype=float)
    d_n = np.full_like(dbs_arr,  np.nan, dtype=float)
    if np.any(valid):
        s_min, s_max = np.nanmin(silh_arr[valid]), np.nanmax(silh_arr[valid])
        d_min, d_max = np.nanmin(dbs_arr[valid]),  np.nanmax(dbs_arr[valid])
        s_den = (s_max - s_min) if (s_max - s_min) > 0 else 1.0
        d_den = (d_max - d_min) if (d_max - d_min) > 0 else 1.0
        s_n[valid] = (silh_arr[valid] - s_min) / s_den        # target 1
        d_n[valid] = (dbs_arr[valid]  - d_min)  / d_den       # target 0
    return s_n, d_n, valid

def optimize_w_weights_by_stability_fast(
    ks_list: list[int],
    res,                     # KScanResult from scan_k (already computed)
    *,
    X: np.ndarray,           # precomputed standardized float32 features
    alphas=(0.25, 0.5, 1.0, 2.0, 4.0),
    betas=(0.25, 0.5, 1.0, 2.0, 4.0),
    random_state: int = 42,
    n_init: int | str = "auto",
    bootstraps: int = 8,
    sample_frac: float = 0.8,
    n_jobs_stability: int = 8,   # parallelize stability over k
):
    # 1) Normalize metrics once
    s_n, d_n, valid = _normalize_metrics(res.silhouette, res.davies_bouldin)
    if not np.any(valid):
        raise ValueError("No valid (silhouette, DB) pairs to optimize over.")

    ks_arr = np.array(ks_list)
    valid_idx = np.where(valid)[0]
    ks_valid = ks_arr[valid_idx]

    rng = np.random.RandomState(random_state)

    def top_k_candidates_by_wdist(res, m=5):
        print("Selecting top k candidates by W-distance...")
    # normalize once
        def _norm(arr):
            a = np.array([np.nan if v is None else float(v) for v in arr], dtype=float)
            v = np.isfinite(a)
            out = np.full_like(a, np.nan)
            if np.any(v):
                mn, mx = np.nanmin(a[v]), np.nanmax(a[v])
                den = (mx - mn) if (mx - mn) > 0 else 1.0
                out[v] = (a[v] - mn) / den
            return out, v

        s_n, v_s = _norm(res.silhouette)     # higher→1
        d_n, v_d = _norm(res.davies_bouldin) # lower→0
        valid = v_s & v_d
        ks = np.array(res.ks)
        one_minus_s = 1 - s_n[valid]
        d_vec = d_n[valid]
        wdist = np.sqrt(one_minus_s**2 + d_vec**2)
        ks_valid = ks[valid]
        order = np.argsort(wdist)
        return list(ks_valid[order[:m]]), {"wdist": wdist, "ks_valid": ks_valid}



    def stability_for_k(k: int) -> float:
        print(f"Computing stability for k={k}...")
        with threadpool_limits(limits=1):  # avoid oversubscription
            km0 = KMeans(n_clusters=k, random_state=random_state, n_init=n_init, algorithm="elkan")
            L0 = km0.fit_predict(X)
            n = X.shape[0]
            aris = []
            for b in range(bootstraps):
                idx = rng.choice(n, size=int(max(2, sample_frac * n)), replace=True)
                Xb = X[idx]
                kmb = KMeans(n_clusters=k, random_state=random_state + b + 1, n_init=n_init, algorithm="elkan")
                Lb = kmb.fit_predict(Xb)
                L0b = km0.predict(Xb)
                aris.append(adjusted_rand_score(L0b, Lb))
            return float(np.mean(aris)) if aris else np.nan

    # 2) Compute stability
    if len(ks_valid) == 1:
        cand_ks, _ = top_k_candidates_by_wdist(res, m=5)
        stability_scores = {int(cand_ks[0]): stability_for_k(int(cand_ks[0]))}
    else:
        print(f"B Computing stability for {len(ks_valid)} k values in parallel...")
        stab_vals = Parallel(n_jobs=n_jobs_stability, prefer="threads", require="sharedmem", verbose=0)(
            delayed(stability_for_k)(int(k)) for k in ks_valid
        )
        stability_scores = {int(k): float(v) for k, v in zip(ks_valid, stab_vals)}

    # 3) Grid search in O(len(α) * len(β) * |K|) with just vector ops
    tried = []
    best = {"alpha": None, "beta": None, "k": None, "score": -np.inf}

    # Precompute vectors used in distance for valid ks
    one_minus_s = 1.0 - s_n[valid]  # shape: |valid|
    d_vec = d_n[valid]

    for a in alphas:
        for b in betas:
            # wdist over valid ks
            wdist = np.sqrt((a * one_minus_s)**2 + (b * d_vec)**2)
            k_star = int(ks_valid[int(np.argmin(wdist))])
            score = float(stability_scores.get(k_star, np.nan))
            tried.append({"alpha": a, "beta": b, "k": k_star, "stability": score})
            if np.isfinite(score) and score > best["score"]:
                best.update({"alpha": a, "beta": b, "k": k_star, "score": score})

    summary = {"trials": tried, "best": best, "stability_by_k": stability_scores}
    print(f"Best stability {best['score']:.4f} at alpha={best['alpha']}, beta={best['beta']}, k={best['k']}")
    return best["alpha"], best["beta"], best["k"], summary


In [12]:

!ls drive/

# 1) Load
# Usage - replace the original code

PARQUET_PATH = "./parquet/spy_historical_2005_2023.parquet"

numeric_features = [
    'impl_volatility', 'delta', 'theta', 'vega',
    'moneyness', 'volume', 'open_interest',
    'vol', 'volume_ma5', 'prc',
    'price_diff_1d',
    'price_diff_2d',
    'price_diff_3d',
    'price_diff_5d',
    'price_diff_8d',
    'price_diff_34d'
]


sample_df = lazy_sample_parquet(PARQUET_PATH, fraction=0.001, columns=numeric_features)

# 2) PCA
result, pca_obj, pca_cols = apply_pca_preprocessing(sample_df, numeric_features)

# 3) Precompute X once (reused by scan + final fit)
feature_set = pca_cols  # or raw numeric_features
X = _prep_features(result, feature_set, standardize=True, dtype=np.float32)

# 4) Auto k-range
k_range = auto_k_range(result, feature_set,max_cap=9)

res = scan_k(df=None, feature_cols=None, k_range=k_range, X=X,
             n_jobs=8, silhouette_sample_size=5000, use_minibatch=True, mbk_max_iter=100)

alpha, beta, k_star, info = optimize_w_weights_by_stability_fast(
    ks_list=list(res.ks),
    res=res,
    X=X,
    alphas=(0.25, 0.5, 1.0, 2.0, 4.0),
    betas=(0.25, 0.5, 1.0, 2.0, 4.0),
    n_init=20,
    bootstraps=6,          # you can lower this; it’s now multiplied by |K|, not by |K|*|grid|
    sample_frac=0.8,
    n_jobs_stability=8,
)
print(f"Stability-picked: α={alpha}, β={beta} → k*={k_star} (mean ARI={info['best']['score']:.3f})")

# 6) (Optional) plot diagnostics reusing X
# kscan = scan_k(df=None, feature_cols=None, k_range=k_range, X=X, w_alpha=alpha, w_beta=beta, n_init=20, n_jobs=8, silhouette_sample_size=5000)
# plot_k_scan(kscan)

# 7) Final fit from X
clustered_auto_df = perform_clustering_from_X(X, result, n_clusters=k_star, n_init=20)

# 8) Analysis (all-Polars; no copies)
cluster_stats_auto = analyze_clusters(clustered_auto_df, feature_cols=feature_set, cluster_col="cluster")

# 9) Save once
clustered_auto_df.write_parquet("./parquet/clusted_auto_SPY.parquet")


ls: drive/: No such file or directory
Sampling ≈0.1% of rows (columns=16)
Loaded 52,176 rows out of 52,176,345
PCA reduced 16 → 12 comps (explained=0.950)


ValueError: output array is read-only

In [ ]:
# 1) decide which non-PCA columns you want available
extra_cols = [
    "prc", "volume", "open_interest",
    "impl_volatility", "delta", "theta",  # raw features
    # "date"  # include if you have a timestamp column
]

# 2) build the set of columns to carry into pandas
plot_cols = build_plot_cols(feature_set=pca_cols, label_col="cluster", max_feats=8)
plot_cols += [c for c in extra_cols if c not in plot_cols and c in clustered_auto_df.columns]

# 3) convert to pandas (with our robust helper from before)
df_plot = to_pandas_minimal(clustered_auto_df, plot_cols, label_col="cluster")

# (optional) if you added a timestamp column that came over as object, fix its dtype:
for dtc in ("date", "timestamp", "datetime"):
    if dtc in df_plot.columns and not pd.api.types.is_datetime64_any_dtype(df_plot[dtc]):
        df_plot[dtc] = pd.to_datetime(df_plot[dtc], errors="coerce")

# 4A) let the widget auto-discover axes (will now include your extras)
regime_dashboard_windows(df=df_plot, label_col="cluster")


In [ ]:
# QQQ pipeline

path = "./parquet/qqq_historical_2005_2023.parquet"

# 1) Load
sample_qqq = lazy_sample_parquet(path, fraction=0.001, columns=numeric_features)

# 2) PCA (once)
result_qqq, pca_obj_qqq, pca_cols_qqq = apply_pca_preprocessing(sample_qqq, numeric_features)
# loadings_qqq = pd.DataFrame(pca_obj_qqq.components_, columns=numeric_features)
# loadings_qqq.index = [f'PC{i+1}' for i in range(loadings_qqq.shape[0])]
# print("PCA Loadings (each row = PC, each column = feature weight):")
# display(loadings_qqq.head())

# 3) Choose k-range on reduced space
k_range_qqq = auto_k_range(result_qqq, pca_cols_qqq)

# 4) Precompute features once
X_qqq = _prep_features(result_qqq, pca_cols_qqq, standardize=True, dtype=np.float32)

# 5) Fast scan + weight search (cached stability)
res_qqq = scan_k(df=None, feature_cols=None, k_range=k_range_qqq, X=X_qqq,
                 n_jobs=8, silhouette_sample_size=5000, use_minibatch=True, mbk_max_iter=100)

alpha_qqq, beta_qqq, k_star_qqq, info_qqq = optimize_w_weights_by_stability_fast(
    ks_list=list(res_qqq.ks),
    res=res_qqq,
    X=X_qqq,
    alphas=(0.25, 0.5, 1.0, 2.0, 4.0),
    betas=(0.25, 0.5, 1.0, 2.0, 4.0),
    n_init=20,
    bootstraps=6,
    sample_frac=0.8,
    n_jobs_stability=8,
)
print(f"QQQ: α={alpha_qqq}, β={beta_qqq} → k*={k_star_qqq} (mean ARI={info_qqq['best']['score']:.3f})")

# (Optional) diagnostics using the chosen weights
# plot_k_scan(res_qqq)

# Final clustering from precomputed X
clustered_auto_qqq_df = perform_clustering_from_X(X_qqq, result_qqq, n_clusters=k_star_qqq, n_init=20)

#save clustered_auto_qqq_df to parquet
clustered_auto_qqq_df.write_parquet("../parquet/clustered_auto_qqq_df.parquet")